# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Neel0289/FlyRank-Week1A1/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

## 1. Ranked actions + reason codes

The validated ranking is used as a decision-support queue for SEO/content reviewers. Pages near the top are not treated as automatic recommendations; they are pages that deserve earlier human attention.

Each recommendation includes a reason code so that a reviewer can understand why the page was prioritized.

The main reason codes are:

- `CTR_POSITION_GAP` — the page has meaningful CTR underperformance relative to its search position.
- `HIGH_DEMAND_LOW_CTR` — the page has meaningful search demand but weak click-through rate.
- `TREND_DECLINE` — recent performance trend is negative and may justify investigation.
- `STALE_REFRESH_CANDIDATE` — the page is relatively old or has not been updated recently.
- `ENGAGEMENT_REVIEW` — engagement signals suggest that the page deserves human review.
- `GENERAL_REVIEW` — the model score is high, but no single diagnostic reason dominates.

The action is deliberately phrased as a review action rather than an automatic change.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestRegressor

# --------------------------------------------------
# 1. Load data
# --------------------------------------------------

df = pd.read_csv(
    "../../data/raw/content_refresh_anonymized.csv"
)

# --------------------------------------------------
# 2. Fields used to construct the proxy
#    These are NOT model features.
# --------------------------------------------------

target_cols = [
    "engagement_rate",
    "scroll_rate",
    "trend_pct"
]

# --------------------------------------------------
# 3. Honest model features
#    Matches the Week-6 leakage correction.
# --------------------------------------------------

feature_cols = [
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "engaged_sessions_90d",
    "search_volume",
    "ctr",
    "avg_position",
    "ai_traffic_pct",
    "content_age_days",
    "days_since_last_update"
]

required_cols = (
    feature_cols
    + target_cols
    + ["client_id"]
)

model_df = df.dropna(
    subset=required_cols
).copy()

# --------------------------------------------------
# 4. Grouped client split
# --------------------------------------------------

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(
        model_df,
        groups=model_df["client_id"]
    )
)

train_df = model_df.iloc[train_idx].copy()
test_df = model_df.iloc[test_idx].copy()

# --------------------------------------------------
# 5. Build proxy separately within each split
# --------------------------------------------------

def build_proxy(data):
    return (
        data["engagement_rate"].rank(pct=True) * 0.5
        + data["scroll_rate"].rank(pct=True) * 0.2
        + data["trend_pct"].rank(pct=True) * 0.3
    )

train_df["proxy_relevance"] = build_proxy(train_df)
test_df["proxy_relevance"] = build_proxy(test_df)

# --------------------------------------------------
# 6. Train corrected model
# --------------------------------------------------

rf = RandomForestRegressor(
    n_estimators=25,
    max_depth=8,
    min_samples_leaf=10,
    random_state=42,
    n_jobs=-1
)

rf.fit(
    train_df[feature_cols],
    train_df["proxy_relevance"]
)

test_df["model_score"] = rf.predict(
    test_df[feature_cols]
)

# --------------------------------------------------
# 7. Rank pages
# --------------------------------------------------

queue = test_df.copy()

queue["rank"] = (
    queue["model_score"]
    .rank(
        ascending=False,
        method="first"
    )
    .astype(int)
)

queue = queue.sort_values(
    "rank"
)

print("Ranked pages:", len(queue))
display(
    queue[
        [
            "rank",
            "content_id",
            "model_score",
            "ctr",
            "avg_position",
            "search_volume",
            "trend_pct",
            "content_age_days"
        ]
    ].head(20)
)

# --------------------------------------------------
# 8. Derived diagnostic signals
# --------------------------------------------------

position_ctr_median = (
    queue
    .groupby(
        pd.cut(
            queue["avg_position"],
            bins=[0, 1, 3, 5, 10, 20, np.inf],
            labels=["1", "2-3", "4-5", "6-10", "11-20", "21+"],
            include_lowest=True
        ),
        observed=False
    )["ctr"]
    .median()
)

position_bucket = pd.cut(
    queue["avg_position"],
    bins=[0, 1, 3, 5, 10, 20, np.inf],
    labels=["1", "2-3", "4-5", "6-10", "11-20", "21+"],
    include_lowest=True
)

queue["expected_ctr"] = position_bucket.map(
    position_ctr_median
)

queue["ctr_gap"] = (
    queue["expected_ctr"] - queue["ctr"]
).clip(lower=0)

# --------------------------------------------------
# 9. Reason code
# --------------------------------------------------

def assign_reason(row):

    if (
        row["search_volume"] > 0
        and row["ctr_gap"] > queue["ctr_gap"].median()
    ):
        return "HIGH_DEMAND_LOW_CTR"

    if row["trend_pct"] < 0:
        return "TREND_DECLINE"

    if (
        row["content_age_days"] >= 180
        or row["days_since_last_update"] >= 180
    ):
        return "STALE_REFRESH_CANDIDATE"

    if (
        row["engagement_rate"] < queue["engagement_rate"].median()
    ):
        return "ENGAGEMENT_REVIEW"

    return "GENERAL_REVIEW"

queue["reason_code"] = queue.apply(
    assign_reason,
    axis=1
)

# --------------------------------------------------
# 10. Action mapping
# --------------------------------------------------

action_map = {
    "HIGH_DEMAND_LOW_CTR":
        "Review title, meta description, and search-intent alignment",

    "TREND_DECLINE":
        "Review recent performance decline and check whether content needs refresh",

    "STALE_REFRESH_CANDIDATE":
        "Review content freshness, outdated information, and refresh opportunity",

    "ENGAGEMENT_REVIEW":
        "Review page experience, content usefulness, and engagement signals",

    "GENERAL_REVIEW":
        "Perform general human review before selecting an optimization action"
}

queue["action"] = queue["reason_code"].map(
    action_map
)

# --------------------------------------------------
# 11. What could make the recommendation wrong?
# --------------------------------------------------

queue["what_would_make_it_wrong"] = (
    "The signal may reflect legitimate search intent, "
    "seasonality, measurement limitations, or a page where "
    "the recommended change would not improve the underlying outcome."
)

display(
    queue[
        [
            "rank",
            "content_id",
            "model_score",
            "reason_code",
            "action",
            "what_would_make_it_wrong"
        ]
    ].head(20)
)

Ranked pages: 4564


,rank,content_id,model_score,ctr,avg_position,search_volume,trend_pct,content_age_days
23859,1,content_b954bc4acaad,0.917820,0.00,41.4,3600.0,-60.0,545
5874,2,content_d82e950a6d46,0.896620,0.00,63.5,70.0,29.5,487
24123,3,content_04e97796ac4d,0.893448,0.11,51.4,880.0,-21.5,445
887,4,content_6fd9a41894ee,0.890739,0.02,47.8,10.0,-12.5,445
5395,5,content_ff150bc51704,0.890127,0.29,43.9,390.0,15.5,445
18062,6,content_b92a65abeeab,0.889209,0.01,42.1,10.0,-73.6,445
7744,7,content_b26ac45aca5d,0.888523,0.14,36.8,20.0,-58.2,445
19333,8,content_9f2ca3ad3165,0.888393,0.15,38.4,1900.0,-21.8,445
13273,9,content_eb1633fa9519,0.885928,0.10,40.1,20.0,-25.8,487
14385,10,content_28742531f88c,0.884858,0.15,22.9,30.0,-35.7,445


,rank,content_id,model_score,reason_code,action,what_would_make_it_wrong
23859,1,content_b954bc4acaad,0.917820,HIGH_DEMAND_LOW_CTR,"Review title, meta description, and search-int...",The signal may reflect legitimate search inten...
5874,2,content_d82e950a6d46,0.896620,HIGH_DEMAND_LOW_CTR,"Review title, meta description, and search-int...",The signal may reflect legitimate search inten...
24123,3,content_04e97796ac4d,0.893448,TREND_DECLINE,Review recent performance decline and check wh...,The signal may reflect legitimate search inten...
887,4,content_6fd9a41894ee,0.890739,HIGH_DEMAND_LOW_CTR,"Review title, meta description, and search-int...",The signal may reflect legitimate search inten...
5395,5,content_ff150bc51704,0.890127,STALE_REFRESH_CANDIDATE,"Review content freshness, outdated information...",The signal may reflect legitimate search inten...
18062,6,content_b92a65abeeab,0.889209,HIGH_DEMAND_LOW_CTR,"Review title, meta description, and search-int...",The signal may reflect legitimate search inten...
7744,7,content_b26ac45aca5d,0.888523,TREND_DECLINE,Review recent performance decline and check wh...,The signal may reflect legitimate search inten...
19333,8,content_9f2ca3ad3165,0.888393,TREND_DECLINE,Review recent performance decline and check wh...,The signal may reflect legitimate search inten...
13273,9,content_eb1633fa9519,0.885928,TREND_DECLINE,Review recent performance decline and check wh...,The signal may reflect legitimate search inten...
14385,10,content_28742531f88c,0.884858,TREND_DECLINE,Review recent performance decline and check wh...,The signal may reflect legitimate search inten...


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

## 2. Intended use and limits

This queue is intended for SEO specialists and content reviewers who need to decide which pages should receive attention first.

The score is a prioritization signal, not a prediction of guaranteed traffic growth. The underlying relevance value is a constructed proxy rather than an observed future business outcome.

The queue should be used to focus human attention, not to automatically publish, rewrite, delete, redirect, or otherwise modify content.

The results are also limited by the available anonymized historical data. They may not generalize to clients, topics, search environments, or future conditions that differ from the observed data.

The intended workflow is:

1. Use the ranked queue to identify pages worth reviewing.
2. Read the reason code.
3. Inspect the page and its search context.
4. Decide whether the recommended action is appropriate.
5. Only then make a content change.

In [4]:
print("QUEUE SUMMARY")
print("=" * 50)

print("Total ranked pages:", len(queue))
print(
    "Unique reason codes:",
    queue["reason_code"].nunique()
)

print("\nReason-code distribution:")
display(
    queue["reason_code"]
    .value_counts()
    .rename_axis("reason_code")
    .reset_index(name="pages")
)

print("\nTop 20 score range:")
print(
    queue.head(20)["model_score"].min(),
    "to",
    queue.head(20)["model_score"].max()
)

print("\nMissing values in recommendation fields:")
display(
    queue[
        [
            "model_score",
            "reason_code",
            "action"
        ]
    ]
    .isna()
    .sum()
)

QUEUE SUMMARY
Total ranked pages: 4564
Unique reason codes: 4

Reason-code distribution:


,reason_code,pages
0,TREND_DECLINE,2317
1,HIGH_DEMAND_LOW_CTR,1369
2,STALE_REFRESH_CANDIDATE,587
3,GENERAL_REVIEW,291



Top 20 score range:
0.874697703274871 to 0.9178198467937606

Missing values in recommendation fields:


model_score    0
reason_code    0
action         0
dtype: int64

## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

## 3. Human review + the no-go list

Every recommendation requires human review before action.

A reviewer should check:

- whether the page's search intent actually matches the proposed action;
- whether the observed CTR gap is meaningful for the page's query mix;
- whether a negative trend is temporary, seasonal, or caused by measurement changes;
- whether the content is actually outdated before recommending a refresh;
- whether changing the page could damage useful rankings or existing traffic.

### No-go cases

The system should NOT automatically:

- publish or rewrite content;
- change titles or meta descriptions without review;
- delete or redirect a page;
- change canonical or indexation settings;
- make claims that a recommendation will increase traffic or rankings;
- treat the model score as a guaranteed business outcome.

The model is decision-support only. Final action remains with a human reviewer.

In [5]:
# --------------------------------------------------
# Human-review checklist
# --------------------------------------------------

review_columns = [
    "rank",
    "content_id",
    "model_score",
    "reason_code",
    "action",
    "what_would_make_it_wrong"
]

review_queue = queue.head(20)[review_columns].copy()

review_queue["human_review_required"] = True
review_queue["approved_for_action"] = False

display(review_queue)

print(
    "\nAll recommendations require human review:",
    review_queue["human_review_required"].all()
)

print(
    "Automatic approvals:",
    review_queue["approved_for_action"].sum()
)

,rank,content_id,model_score,reason_code,action,what_would_make_it_wrong,human_review_required,approved_for_action
23859,1,content_b954bc4acaad,0.917820,HIGH_DEMAND_LOW_CTR,"Review title, meta description, and search-int...",The signal may reflect legitimate search inten...,True,False
5874,2,content_d82e950a6d46,0.896620,HIGH_DEMAND_LOW_CTR,"Review title, meta description, and search-int...",The signal may reflect legitimate search inten...,True,False
24123,3,content_04e97796ac4d,0.893448,TREND_DECLINE,Review recent performance decline and check wh...,The signal may reflect legitimate search inten...,True,False
887,4,content_6fd9a41894ee,0.890739,HIGH_DEMAND_LOW_CTR,"Review title, meta description, and search-int...",The signal may reflect legitimate search inten...,True,False
5395,5,content_ff150bc51704,0.890127,STALE_REFRESH_CANDIDATE,"Review content freshness, outdated information...",The signal may reflect legitimate search inten...,True,False
18062,6,content_b92a65abeeab,0.889209,HIGH_DEMAND_LOW_CTR,"Review title, meta description, and search-int...",The signal may reflect legitimate search inten...,True,False
7744,7,content_b26ac45aca5d,0.888523,TREND_DECLINE,Review recent performance decline and check wh...,The signal may reflect legitimate search inten...,True,False
19333,8,content_9f2ca3ad3165,0.888393,TREND_DECLINE,Review recent performance decline and check wh...,The signal may reflect legitimate search inten...,True,False
13273,9,content_eb1633fa9519,0.885928,TREND_DECLINE,Review recent performance decline and check wh...,The signal may reflect legitimate search inten...,True,False
14385,10,content_28742531f88c,0.884858,TREND_DECLINE,Review recent performance decline and check wh...,The signal may reflect legitimate search inten...,True,False



All recommendations require human review: True
Automatic approvals: 0


,metric,value
0,rows_in_queue,4564.000000
1,missing_score_pct,0.000000
2,median_model_score,0.420827
3,p90_model_score,0.776080
4,negative_trend_pct,74.693252
5,stale_180d_pct,65.863278


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

## 4. Monitoring / retrain triggers

The playbook should be treated as a monitored decision-support system rather than a one-time permanent ranking.

### Monitoring triggers

Re-check the queue when:

- important input fields have unusual missingness;
- the distribution of major signals changes substantially;
- the score distribution becomes unusually concentrated;
- content or search behavior changes enough that the current ranking no longer looks representative;
- the model's measured ranking quality falls materially below the validation result when a comparable evaluation set becomes available.

### Refresh / retrain triggers

A model refresh should be considered when new labeled or outcome data becomes available, when the feature distributions materially drift, or when repeated validation shows that the current ranking is no longer useful relative to the simple baseline.

The trigger should lead to re-validation first, not automatic deployment of a new model.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# --------------------------------------------------
# Lightweight monitoring snapshot
# --------------------------------------------------

monitoring = pd.DataFrame({
    "metric": [
        "rows_in_queue",
        "missing_score_pct",
        "median_model_score",
        "p90_model_score",
        "negative_trend_pct",
        "stale_180d_pct"
    ],
    "value": [
        len(queue),
        queue["model_score"].isna().mean() * 100,
        queue["model_score"].median(),
        queue["model_score"].quantile(0.90),
        (queue["trend_pct"] < 0).mean() * 100,
        (
            (
                (queue["content_age_days"] >= 180)
                | (queue["days_since_last_update"] >= 180)
            ).mean() * 100
        )
    ]
})

display(monitoring)

,metric,value
0,rows_in_queue,4564.000000
1,missing_score_pct,0.000000
2,median_model_score,0.420827
3,p90_model_score,0.776080
4,negative_trend_pct,74.693252
5,stale_180d_pct,65.863278


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from pathlib import Path
import json

# --------------------------------------------------
# Output directories
# --------------------------------------------------

output_dir = Path("../../work/outputs")
figure_dir = Path("../../work/figures")

output_dir.mkdir(
    parents=True,
    exist_ok=True
)

figure_dir.mkdir(
    parents=True,
    exist_ok=True
)

# --------------------------------------------------
# Final export columns
# --------------------------------------------------

export_columns = [
    "rank",
    "content_id",
    "model_score",
    "reason_code",
    "action",
    "what_would_make_it_wrong",
    "human_review_required",
    "approved_for_action"
]

final_queue = review_queue[export_columns].copy()

# --------------------------------------------------
# Write queue CSV
# --------------------------------------------------

queue_path = (
    output_dir
    / "action_playbook_queue.csv"
)

final_queue.to_csv(
    queue_path,
    index=False
)

# --------------------------------------------------
# Save monitoring metrics
# --------------------------------------------------

metrics = {
    "rows_ranked": int(len(queue)),
    "top20_rows_exported": int(len(final_queue)),
    "reason_code_count": int(
        queue["reason_code"].nunique()
    ),
    "median_model_score": float(
        queue["model_score"].median()
    )
}

metrics_path = (
    output_dir
    / "action_playbook_metrics.json"
)

with open(
    metrics_path,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        metrics,
        f,
        indent=2
    )

print("Queue exported to:", queue_path)
print("Metrics exported to:", metrics_path)

Queue exported to: ..\..\work\outputs\action_playbook_queue.csv
Metrics exported to: ..\..\work\outputs\action_playbook_metrics.json


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.